----------------------------------

Import the required libraries.

----------------------------------------------------

In [17]:
import numpy as np
import pandas as pd
import pickle
import deepchem as dc
from sklearn.ensemble import RandomForestRegressor
import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import rdMolDescriptors
from rdkit.Chem import Draw
from rdkit.Chem import Descriptors
from rdkit import DataStructs
import selfies as sf
import os
import joblib
from sklearn.preprocessing import LabelEncoder
from statistics import mean
from statistics import stdev
import tensorflow as tf
from tensorflow.python.eager import polymorphic_function
polymorphic_function.reduce_retracing = True
dc_version = dc.__version__
rk_version = Chem.rdBase.rdkitVersion
print(f"DeepChem version: {dc_version}, RDKit version: {rk_version}")
import warnings
warnings.filterwarnings("ignore", message="Skipped loading modules with pytorch-geometric dependency")
warnings.filterwarnings("ignore", message="Skipped loading modules with pytorch-lightning dependency")
warnings.filterwarnings("ignore", message="Skipped loading some Jax models")

DeepChem version: 2.8.0, RDKit version: 2024.09.6


--------------------------------------

Loading a dataset

---------------------------------------------

In [ ]:
input_file = pd.read_pickle(r"C:\Users\moham\Desktop\Thesis start\MOHAMMAD 2025\NIST GROUPS\NA\combined_dataset_NA.pkl")

---------------------


Goal of this script in the next window:
- The full dataset was saved as multiple .pkl subset files (not one single file).
- To train the models, we first need one complete dataset.
- This script loads all subset .pkl files, combines them into one DataFrame, and saves it as one file.

Why we need it:
- Stage (Binary classification model) 1: train a binary classifier to predict A vs NA.
- stage (NA models and A models) 2: Train models .



---------------------------

In [18]:

input_folder = r"C:\Users\moham\Desktop\Thesis start\NA&A"
output_file = "combined_dataset_A&NA.pkl"

pkl_files = [f for f in os.listdir(input_folder) if f.endswith(".pkl")]
all_data = []

print(f"🔍 Found {len(pkl_files)} .pkl files in folder.")

for file_name in pkl_files:
    file_path = os.path.join(input_folder, file_name)
    try:
        with open(file_path, 'rb') as f:
            data = pickle.load(f, encoding="latin1")  # Safely load legacy .pkl
            if isinstance(data, pd.DataFrame):
                all_data.append(data)
                print(f"✅ Loaded: {file_name} (rows: {data.shape[0]})")
            else:
                print(f"⚠️ Skipped {file_name} — not a DataFrame")
    except Exception as e:
        print(f"❌ Failed to load {file_name}: {e}")

# Final combination
if all_data:
    combined_df = pd.concat(all_data, ignore_index=True)
    output_path = os.path.join(input_folder, output_file)
    with open(output_path, 'wb') as f:
        pickle.dump(combined_df, f)
    print(f"\n✅ Combined {len(all_data)} files into:\n{output_path}")
    print(f"🧮 Total rows: {combined_df.shape[0]}")
else:
    print("🚫 No valid DataFrames were loaded. Nothing to combine.")


🔍 Found 2 .pkl files in folder.
✅ Loaded: A.pkl (rows: 115578)
✅ Loaded: NA.pkl (rows: 66153)

✅ Combined 2 files into:
C:\Users\moham\Desktop\Thesis start\NA&A\combined_dataset_A&NA.pkl
🧮 Total rows: 181731


-----------------------------------------

Goal of this script in the next window:
Loads the dataset, automatically selects the mass spectrum columns as X and the SMILES column as y.\
Then it removes invalid/fragmented SMILES (cannot convert to SELFIES or contains “.”) and returns the cleaned data + valid SELFIES list.

------------------------------

In [19]:
def clean_dataset(input_path):
    """
    General-purpose cleaner for molecular datasets.
    Extracts valid mass spectra (X) and SMILES (y), removes invalid entries.
    Returns cleaned X, y, and selfies_list for further processing.
    """

    #Load dataset
    ms = pd.read_pickle(input_path)

    #Auto-detect spectrum columns (assume they start with "mz")
    spectrum_cols = [col for col in ms.columns if str(col).startswith("mz")]
    if not spectrum_cols:
        raise ValueError(" No spectrum columns found. Expected columns like 'mz1', 'mz2', ...")

    # Auto-detect SMILES column
    smiles_col = None
    for col in ms.columns:
        if 'smiles' in str(col).lower():
            smiles_col = col
            break
    if smiles_col is None:
        raise ValueError(" No SMILES column found in the dataset.")

    # Extract raw features and targets
    X_raw = ms[spectrum_cols]
    y_raw = ms[smiles_col]

    # Clean SMILES and keep valid SELFIES
    valid_indices = []
    selfies_list = []

    for i, smi in enumerate(y_raw):
        try:
            selfies_str = sf.encoder(smi)
            if '.' not in selfies_str:  # Exclude fragmented molecules
                valid_indices.append(i)
                selfies_list.append(selfies_str)
            else:
                print(f"⛔ Skipping SMILES with dot at index {i}: {smi}")
        except sf.EncoderError:
            print(f"⛔ Invalid SMILES at index {i}: {smi}")

    # 🔹 Filter dataset
    X_clean = X_raw.iloc[valid_indices].reset_index(drop=True)
    y_clean = y_raw.iloc[valid_indices].reset_index(drop=True)

    print(f"\n✅ Cleaned dataset loaded: {len(X_clean)} valid molecules")
    return X_clean, y_clean, selfies_list
# Replace with your actual file path
input_file = r"C:\Users\moham\Desktop\Thesis start\NA&A\combined_dataset_A&NA.pkl"

# Run the cleaner
X, y, selfies = clean_dataset(input_file)


⛔ Invalid SMILES at index 85439: OC(=O)c1cc(ccc1[I]=O)[N+]([O-])=O
⛔ Invalid SMILES at index 107698: OC(=O)c1ccccc1[I]=O
⛔ Invalid SMILES at index 107702: O=[I]c1ccccc1
⛔ Invalid SMILES at index 107703: O=[I]c1ccccc1
⛔ Invalid SMILES at index 107730: CC(=O)O[I](OC(C)=O)c1ccccc1
⛔ Invalid SMILES at index 114573: Fc1c(F)c(F)c([I](OC(=O)C(F)(F)F)OC(=O)C(F)(F)F)c(F)c1F
⛔ Invalid SMILES at index 176844: O=[Cl](=O)(=O)O[Cl](=O)(=O)=O
⛔ Invalid SMILES at index 176845: O=[Cl](=O)(=O)O[Cl](=O)(=O)=O

✅ Cleaned dataset loaded: 181723 valid molecules


------------------------------------------

This code uses RDKit to read each SMILES and assign it to a chemical subgroup class based on: (1) aromatic vs non-aromatic and (2) which elements are present (C, H, O, N, S, Cl, F, …), then it outputs labels like CHO_A or CHNO_NA (otherwise REST or INVALID).

We used this method at the beginning to create the subgroup classes and train the classification models for stage 1 and stage 2, but it did not cover all subgroups in the dataset because RDKit could not correctly analyse/parse every SMILES (or some combinations were not included in the rules). That’s why we switched to the second method: using the .pkl file name as the class label, since the dataset already provides the subgroup name that way.\
Example: \
['C', 'C', 'C', 'C', 'C', 'C', 'O', 'H']
{'C', 'H', 'O'}
if atom_symbols.issubset({'C', 'H', 'O'}):
    return '_CHO_AROMATIC'


--------------------------------------

In [21]:
import pandas as pd
from rdkit import Chem

def assign_group(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return 'INVALID'

    atom_symbols = {atom.GetSymbol() for atom in mol.GetAtoms()}
    aromatic = any(bond.GetIsAromatic() for bond in mol.GetBonds())

    if aromatic:
        if atom_symbols.issubset({'C', 'H'}): return 'CH_A'
        elif atom_symbols.issubset({'C', 'H', 'O'}): return 'CHO_A'
        elif atom_symbols.issubset({'C', 'H', 'N'}): return 'CHN_A'
        elif atom_symbols.issubset({'C', 'H', 'N', 'O'}): return 'CHNO_A'
        elif atom_symbols.issubset({'C', 'H', 'N', 'O', 'S'}): return 'CHNOS_A'
        elif atom_symbols.issubset({'C', 'H', 'S'}): return 'CHS_A'
        elif atom_symbols.issubset({'C', 'H', 'N', 'S'}): return 'CHNS_A'
        elif atom_symbols.issubset({'C', 'H', 'O', 'P'}): return 'CHOP_A'
        elif atom_symbols.issubset({'C', 'H', 'I', 'N', 'O'}): return 'CHINO_A'
        elif atom_symbols.issubset({'C', 'Cl', 'H'}): return 'CClH_A'
        elif atom_symbols.issubset({'C', 'H', 'N', 'O', 'Si'}): return 'CHNOSi_A'
        elif atom_symbols.issubset({'C', 'Cl', 'H', 'N', 'S'}): return 'CClHNS_A'
        elif atom_symbols.issubset({'C', 'Cl', 'H', 'N', 'O'}): return 'CClHNO_A'
        elif atom_symbols.issubset({'C', 'F', 'H', 'N', 'O'}): return 'CFHNO_A'
        elif atom_symbols.issubset({'C', 'F', 'H', 'N'}): return 'CFHN_A'
        elif atom_symbols.issubset({'C', 'Cl', 'F', 'H', 'N', 'O'}): return 'CClFHNO_A'
        elif atom_symbols.issubset({'Br', 'C', 'H', 'O'}): return 'BrCHO_A'
        elif atom_symbols.issubset({'C', 'H', 'O', 'S', 'Si'}): return 'CHOSi_A'
        elif atom_symbols.issubset({'C', 'F', 'H', 'N', 'O', 'S'}): return 'CFHNOS_A'
        elif atom_symbols.issubset({'C', 'Cl', 'H', 'N'}): return 'CClHN_A'
        elif atom_symbols.issubset({'C', 'H', 'O', 'S'}): return 'CHOS_A'
        elif atom_symbols.issubset({'C', 'Cl', 'H', 'O'}): return 'CClHO_A'
        elif atom_symbols.issubset({'C', 'Cl', 'H', 'N', 'O', 'S'}): return 'CClHNOS_A'
        elif atom_symbols.issubset({'Br', 'C', 'H', 'N', 'O'}): return 'BrCHNO_A'
        elif atom_symbols.issubset({'C', 'F', 'H', 'O'}): return 'CFHO_A'
        else: return 'REST_A'
    else:
        if atom_symbols.issubset({'C', 'H'}): return 'CH_NA'
        elif atom_symbols.issubset({'C', 'H', 'O'}): return 'CHO_NA'
        elif atom_symbols.issubset({'C', 'H', 'N'}): return 'CHN_NA'
        elif atom_symbols.issubset({'C', 'H', 'N', 'O'}): return 'CHNO_NA'
        elif atom_symbols.issubset({'C', 'H', 'N', 'O', 'S'}): return 'CHNOS_NA'
        elif atom_symbols.issubset({'C', 'H', 'S'}): return 'CHS_NA'
        elif atom_symbols.issubset({'C', 'H', 'O', 'S'}): return 'CHOS_NA'
        elif atom_symbols.issubset({'C', 'H', 'O', 'S', 'Si'}): return 'CHOSi_NA'
        elif atom_symbols.issubset({'C', 'Cl', 'H', 'O'}): return 'CClHO_NA'
        elif atom_symbols.issubset({'Br', 'C', 'H', 'O'}): return 'BrCHO_NA'
        elif atom_symbols.issubset({'C', 'F', 'H', 'O'}): return 'CFHO_NA'
        elif atom_symbols.issubset({'C', 'H', 'O', 'P'}): return 'CHOP_NA'
        elif atom_symbols.issubset({'C', 'F', 'H', 'N', 'O'}): return 'CFHNO_NA'
        elif atom_symbols.issubset({'C', 'Cl', 'H'}): return 'CClH_NA'
        elif atom_symbols.issubset({'C', 'Cl', 'H', 'N', 'O'}): return 'CClHNO_NA'
        elif atom_symbols.issubset({'C', 'H', 'N', 'S'}): return 'CHNS_NA'
        else: return 'REST_NA'

df = pd.read_pickle(r"C:\Users\moham\Desktop\Thesis start\NA&A\combined_dataset_A&NA.pkl")

y = df["SMILES"]

group_labels = y.apply(assign_group)

# results
print(group_labels.value_counts())
df["group_label"] = group_labels
df.to_pickle(r"C:\Users\moham\Desktop\Thesis start\NA&A\_newfile.pkl")


[13:41:10] Conflicting single bond directions around double bond at index 4.
[13:41:10]   BondStereo set to STEREONONE and single bond directions set to NONE.


SMILES
CHNO_A       37640
CHO_NA       25308
CHNOS_A      15933
CHO_A        14561
CHNO_NA      13564
CClHNO_A      9396
CFHNO_A       8788
CHN_A         5957
CHNOS_NA      5006
REST_A        4658
CH_NA         4526
REST_NA       4200
CHOSi_NA      3071
CHN_NA        2905
BrCHNO_A      2781
CClHO_NA      2468
CClHNS_A      2355
CClHNOS_A     2319
CHNOSi_A      2140
CFHNOS_A      1959
CH_A          1906
CClFHNO_A     1535
BrCHO_A       1504
BrCHO_NA      1494
CFHO_NA       1121
CFHNO_NA       891
CHINO_A        871
CHOP_NA        857
CClHNO_NA      742
CHOP_A         631
CClH_A         601
CHOSi_A         43
Name: count, dtype: int64


-------------------------------------


Second labeling method (used in this project):

At first, we tried to create chemical subgroup classes using RDKit by analyzing the SMILES\
(aromatic/non-aromatic + detected elements). However, this RDKit-based rule method did not\
cover all chemical subgroups present in the dataset (some SMILES could not be parsed or did\
not match the predefined rules).

Therefore, we used a second method: we take the subgroup label directly from the .pkl file\
name. Each .pkl file represents one chemical subgroup, so we add a new column 'y' with that\
label, then combine all files into one labeled dataset for training the subgroup classifier.



-----------------------------------------

In [22]:
import os
import pandas as pd

def combine_group_labeled_subsets(base_dir, output_file):
    all_dfs = []

    for fname in os.listdir(base_dir):
        if not fname.endswith('.pkl'):
            continue

        group_label = fname.replace('.pkl', '') 
        file_path = os.path.join(base_dir, fname)

        try:
            df = pd.read_pickle(file_path)
        except Exception as e:
            print(f"⚠️ Failed to read {fname}: {e}")
            continue

        # Add new column 'y' with the group label the new target column to train the models
        df['y'] = group_label

        all_dfs.append(df)

        print(f"✅ Processed {fname}: {len(df)} molecules")

    # Combine everything
    if not all_dfs:
        print("❌ No valid dataframes were loaded.")
        return

    combined_df = pd.concat(all_dfs, ignore_index=True)
    combined_df.to_pickle(output_file)

    print(f"\n📦 Combined dataset saved to: {output_file}")
    print(f"🧪 Total molecules: {len(combined_df)}")
    print("📊 Group distribution:")
    print(combined_df['y'].value_counts())

base_dir = r"C:\Users\moham\Desktop\Thesis start\na" 
output_file = os.path.join(base_dir, "combined_labeled_dataset_NA.pkl")
combine_group_labeled_subsets(base_dir, output_file)


✅ Processed BrCHO_NA.pkl: 1019 molecules
✅ Processed CClHNO_NA.pkl: 610 molecules
✅ Processed CClHO_NA.pkl: 1823 molecules
✅ Processed CClH_NA.pkl: 613 molecules
✅ Processed CFHNO_NA.pkl: 757 molecules
✅ Processed CFHO_NA.pkl: 898 molecules
✅ Processed CHNOS_NA.pkl: 1491 molecules
✅ Processed CHNO_NA.pkl: 13556 molecules
✅ Processed CHNS_NA.pkl: 602 molecules
✅ Processed CHN_NA.pkl: 2891 molecules
✅ Processed CHOP_NA.pkl: 765 molecules
✅ Processed CHOSi_NA.pkl: 2783 molecules
✅ Processed CHOS_NA.pkl: 1877 molecules
✅ Processed CHO_NA--.pkl: 25295 molecules
✅ Processed CHS_NA.pkl: 1022 molecules
✅ Processed CH_NA.pkl: 4526 molecules
✅ Processed na.pkl: 66153 molecules
✅ Processed REST_NA.pkl: 5625 molecules

📦 Combined dataset saved to: C:\Users\moham\Desktop\Thesis start\na\combined_labeled_dataset_NA.pkl
🧪 Total molecules: 132306
📊 Group distribution:
y
na           66153
CHO_NA--     25295
CHNO_NA      13556
REST_NA       5625
CH_NA         4526
CHN_NA        2891
CHOSi_NA      2783


-------------------------

Reload the new  dataset that has the new  column 

------------------------------------

In [23]:
input_file = pd.read_pickle(r"C:\Users\moham\Desktop\Thesis start\na\combined_labeled_dataset_NA.pkl")
input_file

,Name,Form,Mw,Cas,mz1,mz2,mz3,mz4,mz5,mz6,...,mz595,mz596,mz597,mz598,mz599,mz600,Filter_Form,SMILES,AtomGroup,y
0,"2-Butanol, 3-bromo-, acetate",C6H11BrO2,194,5798-81-2,0,0,0,0,0,0,...,0,0,0,0,0,0,True,CC(Br)C(C)OC(C)=O,BrCHO,BrCHO_NA
1,"4-Bromo-3-methyl-butane-1,3-diol",C5H11BrO2,182,NaN,0,0,0,0,0,0,...,0,0,0,0,0,0,True,CC(O)(CBr)CCO,BrCHO,BrCHO_NA
2,"3,4-Dibromohex-3-ene-2,5-diol",C6H10Br2O2,272,31556-90-8,0,0,0,0,0,0,...,0,0,0,0,0,0,True,CC(O)C(Br)=C(Br)C(C)O,BrCHO,BrCHO_NA
3,"2-Propen-1-ol, 2-bromo-, acetate",C5H7BrO2,178,63915-88-8,0,0,0,0,0,0,...,0,0,0,0,0,0,True,CC(=O)OCC(Br)=C,BrCHO,BrCHO_NA
4,"2-Hexanone, 6-bromo-",C6H11BrO,178,10226-29-6,0,0,0,0,0,0,...,0,0,0,0,0,0,True,CC(=O)CCCCBr,BrCHO,BrCHO_NA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
132301,"1,3,5,2,4,6-Triazatriphosphorine, 2,2,4,4,6,6-...",Cl6N3P3,345,940-71-6,0,0,0,0,0,0,...,0,0,0,0,0,0,True,Cl[P]1(=N[P](=N[P](=N1)(Cl)Cl)(Cl)Cl)Cl,ClNP,REST_NA
132302,"1,3,5,7,2,4,6,8-Tetrazatetraphosphocine, 2,2,4...",Cl8N4P4,460,2950-45-0,0,0,0,0,0,0,...,0,0,0,0,0,0,True,Cl[P]1(=N[P](=N[P](=N[P](=N1)(Cl)Cl)(Cl)Cl)(Cl...,ClNP,REST_NA
132303,"3AH,6AH,9aH-1,3,4,6,7,9,9b-heptaaza-2,3a,5,6a,...",Cl9N7P6,599,33992-37-9,0,0,0,0,0,0,...,0,0,0,0,262,0,True,Cl[P]1(=N[P]2(=N[P](=N[P]3(=N[P](=N[P](=N1)(Cl...,ClNP,REST_NA
132304,Bis(trifluoromethyl)iodophosphine,C2F6IP,296,359-64-8,0,0,0,0,0,0,...,0,0,0,0,0,0,True,FC(F)(F)P(I)C(F)(F)F,CFIP,REST_NA


Purpose:
This function prepares the combined dataset for training a classification model.

What it does:
1) Automatically selects the spectral feature columns (mz* or numeric columns) as X.
2) Uses the label column ('y' if available, otherwise 'label') as the target.
3) Encodes the string labels into integers (needed for ML models).
4) Creates a DeepChem dataset and performs a stratified split:
       70% train / 10% validation / 20% test (keeping class distribution balanced).
5) Saves the resulting splits (X_train, y_train, X_valid, y_valid, X_test, y_test)
       and the fitted LabelEncoder to output_dir for reuse later.


In [24]:
def split_dataset_from_combined_df(df, seed=42, output_dir="saved_splits"):

    os.makedirs(output_dir, exist_ok=True)
    spectrum_cols = [col for col in df.columns if str(col).startswith("mz")]
    if not spectrum_cols:
        raise ValueError(
            "No spectrum columns found. Expected columns like mz1, mz2, ...\n"
            f"Available columns (first 30): {list(df.columns[:30])}"
        )

    if "y" not in df.columns:
        raise ValueError(
            "No label column 'y' found in the dataset.\n"
            "Make sure you created the labels first (e.g., from file names) and saved a combined_labeled_dataset."
        )

    X_array = df[spectrum_cols].to_numpy(dtype=np.float32)
    y_array = df["y"].astype(str).to_numpy()

    le = LabelEncoder()
    y_encoded = le.fit_transform(y_array)
    dataset = dc.data.NumpyDataset(X_array, y_encoded)

    # ---- Stratified split ----
    splitter = dc.splits.RandomStratifiedSplitter()
    train_ds, valid_ds, test_ds = splitter.train_valid_test_split(
        dataset=dataset,
        frac_train=0.7,
        frac_valid=0.1,
        frac_test=0.2,
        seed=seed
    )
    files_to_save = {
        "X_train.pkl": train_ds.X,
        "y_train.pkl": train_ds.y,
        "X_valid.pkl": valid_ds.X,
        "y_valid.pkl": valid_ds.y,
        "X_test.pkl": test_ds.X,
        "y_test.pkl": test_ds.y,
        "label_encoder.pkl": le
    }
    for name, obj in files_to_save.items():
        save_path = os.path.join(output_dir, name)
        joblib.dump(obj, save_path)
        print(f"✅ Saved: {save_path}")
    print("\n✅ Done!")
    print(f"Output folder: {os.path.abspath(output_dir)}")
    print(f"X shape: {X_array.shape}")
    print(f"Number of classes: {len(le.classes_)}")
    print(f"Classes: {list(le.classes_)}")
    return train_ds, valid_ds, test_ds, le
combined_file = r"C:\Users\moham\Desktop\Thesis start\na\combined_labeled_dataset_NA.pkl"
df = pd.read_pickle(combined_file)
output_dir = r"C:\Users\moham\Desktop\Thesis start\code from lazar\Featurization_group_classification\saved_splits2"
train_ds, valid_ds, test_ds, label_encoder = split_dataset_from_combined_df(
    df,
    seed=42,
    output_dir=output_dir
)

✅ Saved: C:\Users\moham\Desktop\Thesis start\code from lazar\Featurization_group_classification\saved_splits2\X_train.pkl
✅ Saved: C:\Users\moham\Desktop\Thesis start\code from lazar\Featurization_group_classification\saved_splits2\y_train.pkl
✅ Saved: C:\Users\moham\Desktop\Thesis start\code from lazar\Featurization_group_classification\saved_splits2\X_valid.pkl
✅ Saved: C:\Users\moham\Desktop\Thesis start\code from lazar\Featurization_group_classification\saved_splits2\y_valid.pkl
✅ Saved: C:\Users\moham\Desktop\Thesis start\code from lazar\Featurization_group_classification\saved_splits2\X_test.pkl
✅ Saved: C:\Users\moham\Desktop\Thesis start\code from lazar\Featurization_group_classification\saved_splits2\y_test.pkl
✅ Saved: C:\Users\moham\Desktop\Thesis start\code from lazar\Featurization_group_classification\saved_splits2\label_encoder.pkl

✅ Done!
Output folder: C:\Users\moham\Desktop\Thesis start\code from lazar\Featurization_group_classification\saved_splits2
X shape: (132306,

--------------------------------

Purpose:
Prepare data for K-Fold cross-validation (NO train/valid/test split).

What it does:
- Selects mz* columns as X (spectra features)
- Takes 'y' (or label_col) as labels
- Encodes labels to integers using LabelEncoder
- Saves: X_array.pkl, y_array.pkl, label_encoder.pkl



-----------------------------

In [ ]:
def save_kfold_ready_arrays_from_df(df, output_dir, label_col="y"):

    os.makedirs(output_dir, exist_ok=True)

    spectrum_cols = [c for c in df.columns if str(c).startswith("mz")]
    if not spectrum_cols:
        raise ValueError(
            " No mz* columns found (expected mz1, mz2, ...). "
            f"First columns: {list(df.columns[:30])}"
        )

    if label_col not in df.columns:
        raise ValueError(f" Label column '{label_col}' not found in df columns.")


    X = df[spectrum_cols].to_numpy(dtype=np.float32)
    y = df[label_col].astype(str).to_numpy()

    le = LabelEncoder()
    y_encoded = le.fit_transform(y)

    joblib.dump(X, os.path.join(output_dir, "X_array.pkl"))
    joblib.dump(y_encoded, os.path.join(output_dir, "y_array.pkl"))
    joblib.dump(le, os.path.join(output_dir, "label_encoder.pkl"))

    print("✅ Saved K-Fold ready files to:", os.path.abspath(output_dir))
    print("   X shape:", X.shape)
    print("   y shape:", y_encoded.shape)
    print("   #classes:", len(le.classes_))
    print("   Classes:", list(le.classes_))


combined_file = r"C:\Users\moham\Desktop\Thesis start\na\combined_labeled_dataset_NA.pkl"
df = pd.read_pickle(combined_file)

output_dir = r"C:\Users\moham\Desktop\Thesis start\code from lazar\Featurization_group_classification\processed_data_kfold"
save_kfold_ready_arrays_from_df(df, output_dir)


✅ Saved K-Fold ready files to: C:\Users\moham\Desktop\Thesis start\code from lazar\Featurization_group_classification\processed_data_kfold
   X shape: (132306, 600)
   y shape: (132306,)
   #classes: 18
   Classes: ['BrCHO_NA', 'CClHNO_NA', 'CClHO_NA', 'CClH_NA', 'CFHNO_NA', 'CFHO_NA', 'CHNOS_NA', 'CHNO_NA', 'CHNS_NA', 'CHN_NA', 'CHOP_NA', 'CHOS_NA', 'CHOSi_NA', 'CHO_NA--', 'CHS_NA', 'CH_NA', 'REST_NA', 'na']


: 

-----------------------